In [ ]:
# Encoder–Decoder, Teacher Forcing, Beam Search
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part3/11-encoder-decoder.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part3').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part3')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable `make_date` helper.
3. Implement the two date languages, generated on the fly.
4. Report or visualize the measured result.

In [ ]:
import random, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# [1]
MONTHS = ["january", "february", "march", "april", "may", "june", "july",
          "august", "september", "october", "november", "december"]
DAYS = dict(zip(MONTHS, [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]))

# [2]
def make_date(rng: random.Random) -> tuple[str, str]:
    mo = rng.randrange(12)
    d = rng.randrange(1, DAYS[MONTHS[mo]] + 1)
    y = rng.randrange(1950, 2026)
    iso = f"{y:04d}-{mo+1:02d}-{d:02d}"
    style = rng.random()
    if style < 0.35:
        src = f"{MONTHS[mo]} {d}, {y}"
    elif style < 0.6:
        src = f"{d} {MONTHS[mo]} {y}"
    elif rng.random() < 0.7:
        src = f"{mo+1:02d}/{d:02d}/{y}"            # US convention: mm/dd
    else:
        src = f"{d:02d}/{mo+1:02d}/{y}"            # EU convention: dd/mm
    return src, iso

rng = random.Random(6050)
pairs, seen_sources = [], set()
# [3]
while len(pairs) < 9000:
    pair = make_date(rng)
    if pair[0] not in seen_sources:
        seen_sources.add(pair[0])
        pairs.append(pair)
train_pairs = pairs[:8000]
valid_pairs = pairs[8000:8500]
test_pairs = pairs[8500:]
# [4]
print(*pairs[:4], sep="\n")
train_sources = {s for s, _ in train_pairs}
valid_sources = {s for s, _ in valid_pairs}
test_sources = {s for s, _ in test_pairs}
print("exact-source overlaps: "
      f"train/valid {len(train_sources & valid_sources)}, "
      f"train/test {len(train_sources & test_sources)}, "
      f"valid/test {len(valid_sources & test_sources)}")

PAD, BOS, EOS = 0, 1, 2                             # special tokens first
SRC = sorted(set("".join(s for s, _ in pairs)))
TGT = sorted(set("".join(t for _, t in pairs)))
src_stoi = {c: i + 3 for i, c in enumerate(SRC)}
tgt_stoi = {c: i + 3 for i, c in enumerate(TGT)}
tgt_itos = {i: c for c, i in tgt_stoi.items()}
V_src, V_tgt = len(SRC) + 3, len(TGT) + 3
print(f"source vocab {V_src}, target vocab {V_tgt}")

**Plan**

1. Encoder, decoder, and the handoff.

In [ ]:
# [1]
class Seq2Seq(nn.Module):
    def __init__(self, v_src: int, v_tgt: int, embed: int = 32,
                 hidden: int = 128, packed: bool = True):
        super().__init__()
        self.packed = packed
        self.src_emb = nn.Embedding(v_src, embed, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(v_tgt, embed, padding_idx=PAD)
        self.encoder = nn.LSTM(embed, hidden, batch_first=True)
        self.decoder = nn.LSTM(embed, hidden, batch_first=True)
        self.out = nn.Linear(hidden, v_tgt)

    def encode(
        self, S: torch.Tensor, lengths: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        e = self.src_emb(S)                          # (B, T_src, embed)
        if self.packed:                              # stop at each true length
            e = nn.utils.rnn.pack_padded_sequence(
                e, lengths, batch_first=True, enforce_sorted=False)
        _, state = self.encoder(e)
        return state                                 # (h, c): the context

    def forward(self, S: torch.Tensor, lengths: torch.Tensor,
                T_in: torch.Tensor) -> torch.Tensor:  # teacher-forced pass
        o, _ = self.decoder(self.tgt_emb(T_in), self.encode(S, lengths))
        return self.out(o)                           # (B, T_tgt, V_tgt) logits

**Plan**

1. Define the reusable helpers: `batchify`, `train_seq2seq`, and `translate`.
2. Define the reusable helpers: `exact_match` and `unambiguous`.
3. Prepare the inputs and fixed settings for the example.
4. Train twice — naive right-padding vs. packed sequences.
5. Report or visualize the measured result.

In [ ]:
# [1]
def batchify(prs: list[tuple[str, str]], idx: list[int]) -> tuple[
    torch.Tensor, torch.Tensor, torch.Tensor
]:
    srcs = [torch.tensor([src_stoi[c] for c in prs[i][0]]) for i in idx]
    lengths = torch.tensor([len(s) for s in srcs])
    tgts = [torch.tensor([BOS] + [tgt_stoi[c] for c in prs[i][1]] + [EOS])
            for i in idx]
    S = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD)
    T = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD)
    return S, lengths, T

def train_seq2seq(mode: str = "tf", packed: bool = True, epochs: int = 25,
                  seed: int = 6050, batch: int = 128,
                  checkpoints: tuple[int, ...] = ()) -> tuple[
                      Seq2Seq, dict[int, float]
                  ]:
    torch.manual_seed(seed)
    model = Seq2Seq(V_src, V_tgt, packed=packed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    n, curve = len(train_pairs), {}
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            S, L, T = batchify(train_pairs, perm[i:i + batch].tolist())
            if mode == "tf":                         # gold rail: one fused call
                logits = model(S, L, T[:, :-1])
            else:                                    # free-running: its own rail
                state = model.encode(S, L)
                tok, outs = T[:, :1], []
                for t in range(T.shape[1] - 1):
                    o, state = model.decoder(model.tgt_emb(tok), state)
                    logit = model.out(o)
                    outs.append(logit)
                    tok = logit.argmax(-1)           # feed its own prediction
                logits = torch.cat(outs, 1)
            loss = F.cross_entropy(logits.reshape(-1, V_tgt),
                                   T[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        if ep in checkpoints:
            curve[ep] = exact_match(model, valid_unamb[:400])
    return model, curve

@torch.no_grad()
def translate(model: Seq2Seq, src: str, maxlen: int = 12) -> str:
    model.eval()
    S = torch.tensor([[src_stoi[c] for c in src]])
    state = model.encode(S, torch.tensor([len(src)]))
    tok, out = torch.tensor([[BOS]]), ""
    for _ in range(maxlen):
        o, state = model.decoder(model.tgt_emb(tok), state)
        tok = model.out(o).argmax(-1)                # greedy: best next char
        if tok.item() == EOS:
            break
        out += tgt_itos.get(tok.item(), "?")
    return out

# [2]
@torch.no_grad()
def exact_match(model: Seq2Seq, prs: list[tuple[str, str]]) -> float:
    return sum(translate(model, s) == t for s, t in prs) / len(prs)

def unambiguous(prs: list[tuple[str, str]]) -> list[tuple[str, str]]:
    return [(s, t) for s, t in prs
            if "/" not in s or int(s[:2]) > 12 or int(s[3:5]) > 12]

# [3]
valid_unamb = unambiguous(valid_pairs)
test_unamb = unambiguous(test_pairs)

cps = (2, 4, 6, 8, 12, 16, 20, 25)
# [4]
naive, _ = train_seq2seq(packed=False)
packed, packed_curve = train_seq2seq(packed=True, checkpoints=cps)
# [5]
print(f"naive right-padding:   validation exact match "
      f"{exact_match(naive, valid_unamb):.1%}")
print(f"packed (true lengths): validation exact match "
      f"{exact_match(packed, valid_unamb):.1%}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Measure encoder-state drift through the padding tail.

In [ ]:
# [1]
short_src = min((s for s, _ in valid_pairs), key=len)
ids = torch.tensor([src_stoi[c] for c in short_src])
n_pads = 8
padded_ids = torch.cat([ids, torch.zeros(n_pads, dtype=torch.long)])
with torch.no_grad():
    states, _ = naive.encoder(naive.src_emb(padded_ids[None]))
states = states.squeeze(0)                        # (len + n_pads, hidden)
T = len(ids)
# [2]
drift = (states[T - 1:] - states[T - 1]).norm(dim=1)

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Train the free-running comparison.
3. Audit all three models once on the final test split.

In [ ]:
# [1]
tf_model, tf_curve = packed, packed_curve
# [2]
fr_model, fr_curve = train_seq2seq("fr", checkpoints=cps)
# [3]
print("one final test audit (unambiguous sources):")
for label, candidate in [("naive padding", naive),
                         ("packed / teacher forced", tf_model),
                         ("packed / free-running", fr_model)]:
    print(f"  {label:25s} {exact_match(candidate, test_unamb):.1%}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Implement the residual errors, up close.

In [ ]:
# [1]
errs = [(s, t, translate(tf_model, s))
        for s, t in test_unamb if translate(tf_model, s) != t]
print(f"{len(errs)} errors on {len(test_unamb)} unambiguous test dates; a sample:")
# [2]
for s, t, g in errs[:3]:
    print(f"  {s!r} -> {g!r}   (truth {t})")

**Plan**

1. Beam search, k candidate rails at once.

In [ ]:
# [1]
@torch.no_grad()
def beam_search(model: Seq2Seq, src: str, k: int = 5,
                maxlen: int = 12) -> list[tuple[str, float]]:
    model.eval()
    S = torch.tensor([[src_stoi[c] for c in src]])
    state = model.encode(S, torch.tensor([len(src)]))
    beams = [([], 0.0, state, False)]               # (tokens, logp, state, done)
    for _ in range(maxlen):
        cand = []
        for seq, lp, st, done in beams:
            if done:
                cand.append((seq, lp, st, True)); continue
            tok = torch.tensor([[seq[-1] if seq else BOS]])
            o, st2 = model.decoder(model.tgt_emb(tok), st)
            logp = F.log_softmax(model.out(o)[0, -1], -1)
            top = torch.topk(logp, k)
            for lg, ix in zip(top.values, top.indices):
                cand.append((seq + [ix.item()], lp + lg.item(), st2,
                             ix.item() == EOS))
        beams = sorted(cand, key=lambda b: -b[1])[:k]
        if all(b[3] for b in beams):
            break
    return [("".join(tgt_itos.get(i, "?") for i in seq if i > 2), lp)
            for seq, lp, _, _ in beams]

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Beam search on genuinely ambiguous dates.

In [ ]:
# [1]
amb = [(s, t) for s, t in test_pairs
       if "/" in s and int(s[:2]) <= 12 and int(s[3:5]) <= 12]
# [2]
for s, t in amb[:4]:
    print(f"{s!r}   truth: {t!r}   greedy: {translate(tf_model, s)!r}")
    for text, lp in beam_search(tf_model, s, k=4)[:2]:
        print(f"   beam: {text!r}   joint log score = {lp:.2f}")
    print()

**Plan**

1. How often beam actually changes the answer.
2. Report or visualize the measured result.

In [ ]:
# [1]
sub = random.Random(0).sample(test_pairs, 200)
n_diff = sum(beam_search(tf_model, s, k=5)[0][0] != translate(tf_model, s)
             for s, _ in sub)
# [2]
print(f"beam-5 disagrees with greedy on {n_diff} of {len(sub)} test dates")